# L3 — Orchestrator Harness

## What You'll Learn

- How to create an AgentCore Harness that orchestrates specialist agents discovered via the Registry
- How to configure AgentCore Memory with short-term and long-term extraction strategies
- How to enable CloudWatch observability for the Harness, Memory, and Gateway
- How to invoke the Harness end-to-end and stream the response

## Overview

The Harness is the top-level orchestrator for AnyCompany's customer support system. It holds the conversation with the customer, discovers specialist agents (Order Agent, Refund Agent) through the AgentCore Registry, and delegates tasks to them. It uses Claude Sonnet 4.6 on Bedrock directly — not through LiteLLM — and maintains conversation state via AgentCore Memory (short-term chat history and long-term extracted knowledge).

## Architecture

```
  Customer
       │
       ▼
  ┌─────────────────────────────────────────────────────────┐
  │  AgentCore Harness (anycompany_orchestrator)             │
  │                                                         │
  │  Model: Claude Sonnet 4.6 on Bedrock (direct, no LiteLLM)│
  │  Tools: Code Interpreter, Browser                       │
  │  Memory: AgentCore Memory (short-term + long-term)       │
  └──────────────┬──────────────────────────────────────────┘
                 │  discovers via AgentCore Registry
                 ├──► Order Agent  (A2A, AgentCore Runtime)
                 └──► Refund Agent (A2A, AgentCore Runtime)
```

## Steps

1. Load configuration from SSM
2. Create the IAM execution role for the Harness
3. Create AgentCore Memory
4. Enable observability for Memory and Gateway
5. Create the AgentCore Harness
6. Test the Harness with a sample invocation
7. Save resource IDs to SSM

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- `1_pre-requisites.ipynb` completed — Registry, Gateway, and Cognito IDs must be in SSM
- `1_pluggable_inference_layer.ipynb` completed — `harness_model_id` must be in SSM
- `2_order_agent.ipynb` and `3_refund_agent.ipynb` completed — agents must be registered in the Registry
- `boto3`, `bedrock-agentcore`, and `strands-agents` (installed in the first code cell)

Install required packages.

In [ ]:
%pip install --quiet boto3==1.40.50 bedrock-agentcore==0.1.13 strands-agents==0.5.0

Import libraries, set the region, and initialise AWS clients.

In [ ]:
import boto3
import json
import os
import uuid
import time

# --- Configuration ---
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region

# Clients
from botocore.config import Config
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION,
                                  config=Config(read_timeout=300))
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

## Step 1: Load Configuration from SSM

Fetch Registry and Gateway IDs from SSM — created by `1_pre-requisites.ipynb`.

In [ ]:
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)

def get_ssm(name: str) -> str:
    """Retrieve a single SSM parameter value."""
    return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]

REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
REGISTRY_ARN = get_ssm(f"{SSM_PREFIX}/registry_arn")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")

print(f"Registry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")
print(f"Gateway ID:   {GATEWAY_ID}")
print(f"Gateway ARN:  {GATEWAY_ARN}")

## Step 2: Create the IAM Execution Role

Create the IAM role the Harness assumes at runtime, with permissions for Bedrock, AgentCore Memory, Registry, Gateway, Code Interpreter, and CloudWatch.

In [ ]:
iam = boto3.client("iam")
ROLE_NAME = "AnyCompanyAgentCoreHarnessRole"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "bedrock-agentcore.amazonaws.com",
                    "bedrock.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

# X-Ray tracing requires Resource:* for PutTraceSegments/PutTelemetryRecords per AWS documentation
# CloudWatch metrics require Resource:* but namespace condition limits scope to bedrock-agentcore
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelInvocation",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*"
            ]
        },
        {
            "Sid": "EcrPublicTokenAccess",
            "Effect": "Allow",
            "Action": [
                "ecr-public:GetAuthorizationToken",
                "sts:GetServiceBearerToken"
            ],
            "Resource": "*"
        },
        {
            "Sid": "XRayTracingAccess",
            "Effect": "Allow",
            "Action": [
                "xray:PutTraceSegments",
                "xray:PutTelemetryRecords",
                "xray:GetSamplingRules",
                "xray:GetSamplingTargets"
            ],
            # X-Ray tracing requires Resource: '*' — traces are not resource-specific.
            "Resource": "*"
        },
        {
            "Sid": "CloudWatchMetricsPublish",
            "Effect": "Allow",
            "Action": "cloudwatch:PutMetricData",
            # PutMetricData requires Resource: '*' but is scoped to the
            # bedrock-agentcore namespace via the Condition below.
            "Resource": "*",
            "Condition": {
                "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
            }
        },
        {
            "Sid": "CloudWatchLogsGroup",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:DescribeLogStreams"
            ],
            # Wildcard required for dynamic runtime log groups under /aws/bedrock-agentcore/runtimes/.
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*"
        },
        {
            "Sid": "CloudWatchLogStream",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogStream",
                "logs:PutLogEvents",
                "logs:GetLogEvents"
            ],
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*:log-stream:*"
        },
        {
            "Sid": "CloudWatchLogsDescribeGroups",
            "Effect": "Allow",
            "Action": "logs:DescribeLogGroups",
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:*"
        },
        {
            "Sid": "AgentCoreWorkloadIdentity",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetWorkloadAccessToken",
                "bedrock-agentcore:GetWorkloadAccessTokenForJWT"
            ],
            # Wildcard required for harness workload identities — scoped to harness_* prefix.
            "Resource": [
                f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default",
                f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default/workload-identity/harness_*"
            ]
        },
        {
            "Sid": "AgentCoreBrowserDefault",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:StartBrowserSession",
                "bedrock-agentcore:StopBrowserSession",
                "bedrock-agentcore:GetBrowserSession",
                "bedrock-agentcore:ListBrowserSessions",
                "bedrock-agentcore:UpdateBrowserStream",
                "bedrock-agentcore:ConnectBrowserAutomationStream",
                "bedrock-agentcore:ConnectBrowserLiveViewStream"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:browser/*"
        },
        {
            "Sid": "AgentCoreCodeInterpreterDefault",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:GetCodeInterpreterSession",
                "bedrock-agentcore:ListCodeInterpreterSessions",
                "bedrock-agentcore:InvokeCodeInterpreter"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:code-interpreter/*"
        },
        {
            "Sid": "AgentCoreMemory",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateEvent",
                "bedrock-agentcore:DeleteEvent",
                "bedrock-agentcore:GetEvent",
                "bedrock-agentcore:ListEvents",
                "bedrock-agentcore:RetrieveMemoryRecords"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:memory/*"
        },
        {
        "Sid": "AgentCoreInvokeRuntime",
        "Effect": "Allow",
        "Action": "bedrock-agentcore:InvokeAgentRuntime",
        "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:runtime/*"
        },
        {
            "Sid": "AgentCoreRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:SearchRegistryRecords",
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:ListRegistryRecords"
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:*"
        },
        {
            "Sid": "AgentCoreGatewayAccess",
            "Effect": "Allow",
            "Action": "bedrock-agentcore:InvokeGateway",
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/*"
        }
    ]
}

try:
    role_response = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AnyCompany AgentCore Harness"
    )
    EXECUTION_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {EXECUTION_ROLE_ARN}")

    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="AnyCompanyHarnessPermissions",
        PolicyDocument=json.dumps(permissions_policy)
    )
    print("Attached inline policy.")
    print("Waiting 10s for IAM propagation...")
    time.sleep(10)

except iam.exceptions.EntityAlreadyExistsException:
    EXECUTION_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {EXECUTION_ROLE_ARN}")

print(f"\nExecution Role ARN: {EXECUTION_ROLE_ARN}")

## Step 3: Create AgentCore Memory

Short-term memory (raw conversation events) is automatic. Long-term memory requires explicit strategy configuration — we define three: session summarization, user preference extraction, and semantic fact extraction.

Create the memory resource with three long-term extraction strategies scoped per `actorId`.

In [ ]:
MEMORY_NAME = "anycompany_agent_memory"
print(f"Creating memory: {MEMORY_NAME}")

try:
    memory_response = agentcore_control.create_memory(
        name=MEMORY_NAME,
        description="long-term memory for the AnyCompany orchestrator harness.",
        memoryExecutionRoleArn=EXECUTION_ROLE_ARN,
        eventExpiryDuration=30,  # Events expire after 1 hour
        memoryStrategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "session_summary",
                    "description": "Summarizes each conversation session",
                    # `namespaceTemplates` (not the legacy `namespaces`) supports
                    # variable substitution. {actorId}/{sessionId} ensures each
                    # user's session summaries are isolated from other users'.
                    "namespaceTemplates": ["summaries/{actorId}/{sessionId}"]
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "user_preferences",
                    "description": "Extracts stated user preferences",
                    # Per-user preference scoping. Without {actorId} every
                    # user's preferences would collide in a flat namespace.
                    "namespaceTemplates": ["preferences/{actorId}"]
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "semantic_facts",
                    "description": "Extracts factual knowledge for semantic retrieval",
                    "namespaceTemplates": ["facts/{actorId}"]
                }
            }
        ]
    )

    MEMORY_ID = memory_response["memory"]["id"]
    MEMORY_ARN = memory_response["memory"]["arn"]
    print(f"Created memory: {MEMORY_NAME}")
    print(f"  ID:  {MEMORY_ID}")
    print(f"  ARN: {MEMORY_ARN}")

except (agentcore_control.exceptions.ConflictException, agentcore_control.exceptions.ValidationException):
    print(f"Memory '{MEMORY_NAME}' already exists. Looking up...")
    memories = agentcore_control.list_memories()
    for mem in memories.get("memories", []):
        # list_memories doesn't return name, so get each one
        detail = agentcore_control.get_memory(memoryId=mem["id"])
        if detail["memory"].get("name") == MEMORY_NAME:
            MEMORY_ID = detail["memory"]["id"]
            MEMORY_ARN = detail["memory"]["arn"]
            print(f"  ID:  {MEMORY_ID}")
            print(f"  ARN: {MEMORY_ARN}")
            break
    else:
        raise RuntimeError(f"Could not find memory with name '{MEMORY_NAME}'")

Poll until the Memory resource reaches ACTIVE status.

In [ ]:
# Wait for memory to be ready
import time as _time

print("Waiting for memory to be ready...")
_mem_start = _time.time()
_poll = 0

while True:
    m = agentcore_control.get_memory(memoryId=MEMORY_ID)
    status = m["memory"]["status"]
    _elapsed = _time.time() - _mem_start
    _poll += 1
    if status == "ACTIVE":
        print(f"Memory is ACTIVE after {_elapsed:.1f}s ({_poll} poll(s))")
        break
    elif status == "FAILED":
        reason = m["memory"].get("failureReason", "unknown")
        raise RuntimeError(f"Memory creation failed: {reason}")
    print(f"  [{_elapsed:5.1f}s] Status: {status} — waiting... (poll #{_poll})")
    time.sleep(5)

print(f"\nMemory ID:  {MEMORY_ID}")
print(f"Memory ARN: {MEMORY_ARN}")

## Step 4: Enable Observability

Configure CloudWatch log delivery and tracing for the Memory resource.

In [ ]:
logs_client = boto3.client("logs", region_name=REGION)

MEMORY_LOG_PREFIX = "/aws/vendedlogs/bedrock-agentcore/memory"

for log_type in ["APPLICATION_LOGS"]:
    source_name = f"mem-{MEMORY_ID[:20]}-{log_type.lower()}"
    dest_log_group = f"{MEMORY_LOG_PREFIX}/{log_type}/{MEMORY_ID}"
    dest_name = f"mem-{MEMORY_ID[:20]}-{log_type.lower()}-dest"

    # Check if source already exists
    existing_source = None
    sources = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources:
        if MEMORY_ARN in s.get('resourceArns', []) and s.get('logType') == log_type:
            existing_source = s['name']
            break

    if existing_source:
        source_name = existing_source
        print(f"✓ {log_type} source already exists: {source_name}")
    else:
        try:
            logs_client.put_delivery_source(name=source_name, resourceArn=MEMORY_ARN, logType=log_type)
            print(f"✓ Created {log_type} source: {source_name}")
        except Exception as e:
            print(f"✗ {log_type} source: {e}")

    # Create log group
    try:
        logs_client.create_log_group(logGroupName=dest_log_group)
    except logs_client.exceptions.ResourceAlreadyExistsException:
        pass

    # Create destination
    try:
        logs_client.put_delivery_destination(
            name=dest_name,
            deliveryDestinationConfiguration={
                "destinationResourceArn": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{dest_log_group}"
            },
        )
    except logs_client.exceptions.ConflictException:
        pass

    # Create delivery
    try:
        logs_client.create_delivery(
            deliverySourceName=source_name,
            deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{dest_name}",
        )
        print(f"✓ {log_type} delivery created")
    except logs_client.exceptions.ConflictException:
        print(f"✓ {log_type} delivery already exists")
    except Exception as e:
        print(f"✗ {log_type} delivery: {e}")

# Enable Tracing
trace_source_name = f"mem-{MEMORY_ID[:20]}-traces"
try:
    existing_trace = None
    for s in sources:
        if MEMORY_ARN in s.get('resourceArns', []) and s.get('logType') == 'TRACES':
            existing_trace = s['name']
            break
    if not existing_trace:
        logs_client.put_delivery_source(name=trace_source_name, resourceArn=MEMORY_ARN, logType='TRACES')
        print(f"✓ Trace source created")
    else:
        trace_source_name = existing_trace
        print(f"✓ Trace source exists")

    # Find XRAY destination and create delivery
    xray_dest_arn = None
    dests = logs_client.describe_delivery_destinations().get('deliveryDestinations', [])
    for d in dests:
        if d.get('deliveryDestinationType') == 'XRAY':
            xray_dest_arn = d['arn']
            break
    if xray_dest_arn:
        try:
            logs_client.create_delivery(
                deliverySourceName=trace_source_name,
                deliveryDestinationArn=xray_dest_arn,
            )
            print(f"✓ Tracing delivery created")
        except logs_client.exceptions.ConflictException:
            print(f"✓ Tracing delivery already exists")
    else:
        print("  ⚠ No XRAY destination found. Run agent notebooks first.")
except Exception as e:
    print(f"✗ Tracing: {e}")

print(f"\nMemory observability configured for {MEMORY_ID}")

Configure CloudWatch log delivery and tracing for the Gateway resource.

In [ ]:
# Fetch Gateway config from SSM
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
print(f"Gateway: {GATEWAY_ID}")

GATEWAY_LOG_PREFIX = "/aws/vendedlogs/bedrock-agentcore/gateway"

for log_type in ["APPLICATION_LOGS"]:
    dest_log_group = f"{GATEWAY_LOG_PREFIX}/{log_type}/{GATEWAY_ID}"
    source_name = f"gw-{GATEWAY_ID[:20]}-{log_type.lower()}"
    dest_name = f"gw-{GATEWAY_ID[:20]}-{log_type.lower()}-dest"

    print(f"Configuring {log_type}...")

    # Check if a delivery source already exists for this gateway + log type
    existing_source = None
    sources = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources:
        if GATEWAY_ARN in s.get('resourceArns', []) and s.get('logType') == log_type:
            existing_source = s['name']
            break

    if existing_source:
        source_name = existing_source
        print(f"  ✓ Using existing delivery source: {source_name}")
    else:
        try:
            logs_client.put_delivery_source(name=source_name, resourceArn=GATEWAY_ARN, logType=log_type)
            print(f"  ✓ Created delivery source: {source_name}")
        except Exception as e:
            print(f"  ✗ Delivery source error: {e}")

    # Create log group
    try:
        logs_client.create_log_group(logGroupName=dest_log_group)
        print(f"  ✓ Created log group: {dest_log_group}")
    except logs_client.exceptions.ResourceAlreadyExistsException:
        print(f"  ✓ Log group already exists: {dest_log_group}")

    # Create delivery destination
    try:
        logs_client.put_delivery_destination(
            name=dest_name,
            deliveryDestinationConfiguration={
                "destinationResourceArn": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{dest_log_group}"
            },
        )
        print(f"  ✓ Created delivery destination: {dest_name}")
    except logs_client.exceptions.ConflictException:
        print(f"  ✓ Delivery destination already exists: {dest_name}")

    # Create delivery
    try:
        logs_client.create_delivery(
            deliverySourceName=source_name,
            deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{dest_name}",
        )
        print(f"  ✓ Created delivery")
    except logs_client.exceptions.ConflictException:
        print(f"  ✓ Delivery already exists")
    except Exception as e:
        print(f"  ✗ Delivery error: {e}")

print(f"\nGateway log delivery configured.")

# Enable Tracing for Gateway (reuse XRAY destination from agent deployment)
trace_source_name = f"gw-{GATEWAY_ID[:20]}-traces"
try:
    existing_trace_source = None
    sources_refresh = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources_refresh:
        if GATEWAY_ARN in s.get('resourceArns', []) and s.get('logType') == 'TRACES':
            existing_trace_source = s['name']
            break
    if not existing_trace_source:
        logs_client.put_delivery_source(name=trace_source_name, resourceArn=GATEWAY_ARN, logType='TRACES')
        print(f"✓ Gateway trace source created")
    else:
        trace_source_name = existing_trace_source
        print(f"✓ Gateway trace source exists")

    # Find XRAY destination (auto-created by agent toolkit deployment)
    xray_dest_arn = None
    dests = logs_client.describe_delivery_destinations().get('deliveryDestinations', [])
    for d in dests:
        if d.get('deliveryDestinationType') == 'XRAY':
            xray_dest_arn = d['arn']
            break

    if xray_dest_arn:
        try:
            logs_client.create_delivery(
                deliverySourceName=trace_source_name,
                deliveryDestinationArn=xray_dest_arn,
            )
            print(f"✓ Gateway tracing delivery created")
        except logs_client.exceptions.ConflictException:
            print(f"✓ Gateway tracing delivery already exists")
    else:
        print("  ⚠ No XRAY destination found. Enable tracing manually in the console.")
except Exception as e:
    print(f"✗ Tracing: {e}")

## Step 5: Create the AgentCore Harness

Define the system prompt that governs the Harness's orchestration behaviour.

In [ ]:
SYSTEM_PROMPT = f"""You are the AnyCompany Electronics customer support orchestrator. Your job is to help
customers with order inquiries, refund requests, and product damage claims.

## Distinguishing statements from requests

The customer may STATE preferences or facts ('I prefer email for refund
updates') or REQUEST actions ('what is the status of ORD-10001?'). These
look similar but require different handling:

- A STATEMENT of preference, fact, or contact detail: acknowledge
  naturally in ONE sentence and offer to help with anything else. Do
  NOT trigger order lookups, refund flows, or specialist-agent calls.
  Memory extraction records it automatically; you do not need to act.

- A REQUEST for action ('check my order', 'process a refund', 'what
  is the status of', 'can you'): run the relevant workflow.

If a single message contains both ('My preferred contact is email and
I want to know the status of ORD-10001'), briefly acknowledge the
preference, then handle the request.

The presence of words like 'refund', 'order', or 'product' does NOT
automatically mean the customer wants a workflow. Read the verb:

'I prefer', 'I like', 'I am', 'my X is'      → STATEMENT (acknowledge only)
'can you', 'I want', 'I need', 'check', 'what is'  → REQUEST (run workflow)

## Agent Discovery

You have access to an Agent Registry (ID: {REGISTRY_ID}) that contains specialist agents.
To discover available agents, use the search_agent_registry tool with a natural language query describing what you need.

Example: search_agent_registry(search_query="order management agent")

IMPORTANT: When invoking an agent, use ONLY the `agentRuntimeArn` field from the registry record.
It looks like: arn:aws:bedrock-agentcore:<region>:<account>:runtime/<agent_name>-<id>
Do NOT use endpointUrl, agentRuntimeEndpointUrl, or any URL field as the ARN.
Do NOT pass runtimeSessionId, harnessArn, or any harness parameters when invoking specialist agents.
Those parameters are only for invoke_harness — specialist agents use A2A (HTTP POST + JSON-RPC).
The correct A2A invocation format is:
  - HTTP POST to: https://bedrock-agentcore.<region>.amazonaws.com/runtimes/<escaped_arn>/invocations
  - Body parameter name is `data` (not `payload`, not `request`, not `body`)
  - JSON-RPC method must be `message/send` (NOT tasks/send)
  - The message object MUST include `messageId` (use str(uuid.uuid4()))
  - Read the response from `resp.text` (not `resp.payload`, not `resp.body`)

## How You Work

1. When a customer asks for help, use the search_agent_registry tool to find relevant agents
2. Use the discovered agent information to delegate tasks via the invoke_agent tool
3. For calculations (prorated amounts, partial refunds), use the code_interpreter tool
4. Summarize results clearly and confirm next steps with the customer

## Refund Eligibility Calculation Rules

When a customer requests a refund, follow these steps:

### Step 1: Get order and refund policy details
- Delegate to the Refund Agent to retrieve the refund policy for the customer's tier and order.
- The Refund Agent will return the return window days, refund percentage, and eligibility based on the Knowledge Base.

### Step 2: Calculation for refund
- When calculating refund eligibility,
  1. Use the code_interpreter tool to get the current date: from datetime import date; print(date.today())
  2. Parse the delivery date from the order details and the return window days from the Refund Agent's response.
  3. Calculates days_since_delivery = Find the number of days between current date and date of delivery.
  4. Apply the tier-based rules returned by the Refund Agent — do NOT use hardcoded rules.

### Step 3: Report the result
- State the tier, delivery date, days since delivery, eligibility, and refund amount
- If eligible, mention that they are eligible for return.
- If not eligible, explain why (return window expired)

IMPORTANT: Always use code_interpreter for date math — do NOT calculate in your head.
IMPORTANT: Always get refund policy rules from the Refund Agent — do NOT use hardcoded tier rules.

## Memory

You have access to long-term memory scoped to the current customer.
AgentCore automatically retrieves relevant past context (preferences,
extracted facts, prior session summaries) and injects it before each
turn. The injected context may be EMPTY — that is a real and common
signal, not an error.

Behave accordingly:

- When the customer asks about a preference or fact: answer ONLY from
  what is actually present in the retrieved memory above. If retrieval
  is empty, or the specific detail is not there, say so honestly and
  ask the customer to share. Do NOT confabulate or guess.

- When the customer states a preference, fact, or contact detail,
  acknowledge naturally in one sentence. Extraction is automatic;
  you do not have or need a 'store this' tool.

- Quote or paraphrase what is actually in retrieved memory rather
  than inventing details. If memory says 'Prefers email', you may
  reply 'you mentioned previously you prefer email'. Do not invent
  specifics that are not present in the retrieved text.

- Recently extracted memories may take a few minutes to surface
  across sessions; within a single session, recent messages are
  always visible.

CRITICAL: Never invent memories. If the retrieved context is empty,
the correct answer is 'I don't have that on record yet — could you
share it?' Saying you remember something you do not is a serious
failure.

## Response Style

Since this is a workshop demo, show your reasoning so attendees can
follow the orchestration flow:

- Briefly state what you are doing at each step (e.g., "Searching the
  registry for an order specialist...", "Delegating to the Refund Agent
  to check eligibility...")
- When you delegate to a specialist agent, mention which agent and why
- If a calculation is involved, show the inputs and the math
- Keep it concise — one line per step, not a wall of text
- After showing your work, deliver the final answer clearly to the
  customer, separated by a line break

Do NOT expose raw JSON payloads, full stack traces, or internal error
messages. If a tool call fails, briefly note the issue and retry or
explain what went wrong in plain language.

## Important Rules

- Never make up order information — always query the appropriate agent
- Never approve a refund without checking eligibility
- Only display orders, refund details, or account information belonging to
  the authenticated customer (the customer_id in the current session).
  If a query references a different customer's ID, politely decline and
  explain you can only assist with their own account.
- Use search_agent_registry for discovering agents, invoke_agent for calling them
- Use code_interpreter for math and data analysis
- Be empathetic and professional in all interactions
"""

print("System prompt defined.")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

Create the Harness with Claude Sonnet 4.6, the system prompt, Memory, Code Interpreter, and Browser tools. Updates the existing Harness if it already exists.

In [ ]:
print("Creating AgentCore Harness...")
HARNESS_NAME = "anycompany_orchestrator"

try:
    harness_response = agentcore_control.create_harness(
        harnessName=HARNESS_NAME,
        executionRoleArn=EXECUTION_ROLE_ARN,

        # Model
        model={
            "bedrockModelConfig": {
                # AgentCore's create_harness API is AWS-native and only
                # supports Bedrock. It expects a bare Bedrock model ID.
                # L2 publishes this separately from the worker `model_id`
                # so the orchestrator stays on Bedrock even when workers
                # are swapped to a non-Bedrock provider via LiteLLM.
                "modelId": get_ssm(f"{SSM_PREFIX}/harness_model_id"),
                "maxTokens": 4096,
                "temperature": 0.1
            }
        },

        # System prompt
        systemPrompt=[{"text": SYSTEM_PROMPT}],

        # Tools — browser + code interpreter
        # The harness discovers agents via search_registry_records using its shell/code tools
        tools=[
            {
                "type": "agentcore_browser",
                "name": "browser",
                "config": {
                    "agentCoreBrowser": {}
                }
            },
            {
                "type": "agentcore_code_interpreter",
                "name": "code_interpreter",
                "config": {
                    "agentCoreCodeInterpreter": {}
                }
            }
        ],

        allowedTools=["*"],

        # Memory
        memory={
            "agentCoreMemoryConfiguration": {
                "arn": MEMORY_ARN,
                "messagesCount": 20
            }
        },

        # Execution limits
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,

        tags={
            "project": "anycompany-workshop",
            "layer": "l3-orchestration",
            "role": "orchestrator"
        }
    )

    harness = harness_response["harness"]
    HARNESS_ID = harness["harnessId"]
    HARNESS_ARN = harness["arn"]
    print(f"\nHarness created!")
    print(f"  Name:   {harness['harnessName']}")
    print(f"  ID:     {HARNESS_ID}")
    print(f"  ARN:    {HARNESS_ARN}")
    print(f"  Status: {harness['status']}")

except agentcore_control.exceptions.ConflictException:
    print(f"Harness '{HARNESS_NAME}' already exists. Updating with new system prompt...")
    harnesses = agentcore_control.list_harnesses()
    for h in harnesses.get("harnesses", []):
        if h.get("harnessName") == HARNESS_NAME:
            HARNESS_ID = h["harnessId"]
            HARNESS_ARN = h["arn"]
            break
    else:
        raise RuntimeError(f"Could not find harness with name '{HARNESS_NAME}'")

    agentcore_control.update_harness(
        harnessId=HARNESS_ID,
        executionRoleArn=EXECUTION_ROLE_ARN,
        model={
            "bedrockModelConfig": {
                "modelId": get_ssm(f"{SSM_PREFIX}/harness_model_id"),
                "maxTokens": 4096,
                "temperature": 0.1
            }
        },
        systemPrompt=[{"text": SYSTEM_PROMPT}],
        tools=[
            {"type": "agentcore_browser", "name": "browser", "config": {"agentCoreBrowser": {}}},
            {"type": "agentcore_code_interpreter", "name": "code_interpreter", "config": {"agentCoreCodeInterpreter": {}}}
        ],
        allowedTools=["*"],
        memory={"optionalValue": {"agentCoreMemoryConfiguration": {"arn": MEMORY_ARN, "messagesCount": 20}}},
        maxIterations=20,
        maxTokens=16384,
        timeoutSeconds=300,
    )
    print(f"  Harness updated with new system prompt.")
    print(f"  ID:  {HARNESS_ID}")
    print(f"  ARN: {HARNESS_ARN}")

Poll until the Harness reaches READY status.

In [ ]:
print("Waiting for harness to be ready...")
for _ in range(30):
    harness_info = agentcore_control.get_harness(harnessId=HARNESS_ID)
    status = harness_info["harness"]["status"]
    if status == "READY":
        print(f"Harness is READY!")
        break
    elif "FAIL" in status:
        reason = harness_info["harness"].get("failureReason", "unknown")
        raise RuntimeError(f"Harness creation failed: {status} — {reason}")
    print(f"  Status: {status} — waiting 10s...")
    time.sleep(10)
else:
    raise TimeoutError("Harness did not become ready within 5 minutes.")

print(f"\nHarness ID:  {HARNESS_ID}")
print(f"Harness ARN: {HARNESS_ARN}")

## Step 6: Test the Harness

Invoke the Harness with a sample refund query and stream the response to confirm end-to-end orchestration.

In [ ]:
session_id = str(uuid.uuid4()).upper()
# actorId scopes long-term memory (preferences, facts) per user.
# In the chatbot UI this is the Cognito custom:customer_id claim;
# in this notebook test we hardcode a Gold-tier user.
actor_id = "CUST-789"
print(f"Session ID: {session_id}")
print(f"Actor ID:   {actor_id}\n")

response = agentcore_runtime.invoke_harness(
    harnessArn=HARNESS_ARN,
    runtimeSessionId=session_id,
    actorId=actor_id,
    messages=[{"role": "user", "content": [{"text": "Hi! I want to know my refund for ORD-10001"}]}],
)

# Process the streaming response
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## Step 7: Save Configuration to SSM

Publish Harness and Memory IDs to SSM for use by L5 observability notebooks.

In [ ]:
parameters = {
    f"{SSM_PREFIX}/harness_id": HARNESS_ID,
    f"{SSM_PREFIX}/harness_arn": HARNESS_ARN,
    f"{SSM_PREFIX}/harness_name": HARNESS_NAME,
    f"{SSM_PREFIX}/harness_role_arn": EXECUTION_ROLE_ARN,
    f"{SSM_PREFIX}/memory_id": MEMORY_ID,
    f"{SSM_PREFIX}/memory_arn": MEMORY_ARN,
    f"{SSM_PREFIX}/memory_name": MEMORY_NAME,
}

print(f"Saving {len(parameters)} parameters to SSM under '{SSM_PREFIX}/'\n")

for name, value in parameters.items():
    ssm.put_parameter(
        Name=name,
        Value=value,
        Type="String",
        Overwrite=True,
    )
    print(f"  ✓ {name} = {value}")

print(f"\nAll parameters saved.")

## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| IAM Role | `AnyCompanyAgentCoreHarnessRole` | Execution role for Bedrock, Memory, Registry, Gateway, and CloudWatch |
| AgentCore Memory | `anycompany_agent_memory` | Short-term chat history + long-term summarization, preferences, and semantic facts |
| AgentCore Harness | `anycompany_orchestrator` | Orchestrator running Claude Sonnet 4.6 — discovers and delegates to specialist agents |

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore: harness, memory
- IAM: harness execution role
- Observability: CloudWatch log delivery components and log groups (memory + gateway wiring)
- Configuration: SSM parameters

> Run this cleanup **before** the L3 #1 (`1_setup_resources.ipynb`) cleanup, since this notebook owns the gateway log delivery wiring (the gateway resource itself is owned by L3 #1).

Uncomment and run to delete all resources created in this notebook.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_setup_resources.ipynb) cleanup, since this
##       notebook owns the gateway log delivery wiring (the gateway itself is
##       owned by L3 #1).

# import boto3, os, time
#
    # Note: In production, implement content filtering if responses may contain sensitive user data from memory or tool results.
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# HARNESS_NAME = "anycompany_orchestrator"
# MEMORY_NAME  = "anycompany_agent_memory"
# ROLE_NAME    = "AnyCompanyAgentCoreHarnessRole"
#
# iam               = boto3.client("iam")
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # Resolve IDs/ARNs from SSM, falling back to lookup-by-name (idempotent).
# HARNESS_ID  = _ssm_get("harness_id")
# HARNESS_ARN = _ssm_get("harness_arn")
# MEMORY_ID   = _ssm_get("memory_id")
# MEMORY_ARN  = _ssm_get("memory_arn")
# GATEWAY_ARN = _ssm_get("gateway_arn")
# GATEWAY_ID  = _ssm_get("gateway_id")
#
# if not (HARNESS_ID and HARNESS_ARN):
#     try:
#         for h in agentcore_control.list_harnesses().get("harnesses", []):
#             if h.get("harnessName") == HARNESS_NAME:
#                 HARNESS_ID  = h["harnessId"]
#                 HARNESS_ARN = h["arn"]
#                 break
#     except Exception as e:
#         print(f"List harnesses: {e}")
#
# if not (MEMORY_ID and MEMORY_ARN):
#     try:
#         for mem in agentcore_control.list_memories().get("memories", []):
#             try:
#                 detail = agentcore_control.get_memory(memoryId=mem["id"])
#                 if detail["memory"].get("name") == MEMORY_NAME:
#                     MEMORY_ID  = detail["memory"]["id"]
#                     MEMORY_ARN = detail["memory"]["arn"]
#                     break
#             except Exception:
#                 continue
#     except Exception as e:
#         print(f"List memories: {e}")
#
# # ── Phase 1: Delete Harness (must precede Memory + Role delete) ─────────────
# if HARNESS_ID:
#     try:
#         agentcore_control.delete_harness(harnessId=HARNESS_ID)
#         print(f"Initiated delete of Harness: {HARNESS_ID}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_harness(harnessId=HARNESS_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Harness fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Harness to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Harness {HARNESS_ID} already deleted.")
#     except Exception as e:
#         print(f"Harness delete: {e}")
# else:
#     print(f"Harness {HARNESS_NAME} not found (already deleted).")
#
# # Helper: drain log delivery components for a given resource ARN.
# # IMPORTANT: filter destinations to ours-only (name pattern + skip XRAY type)
# # to avoid deleting the shared XRAY destination used by other agents.
# def _drain_log_delivery(resource_arn, dest_name_prefix):
#     if not resource_arn:
#         return
#     # 1. Find sources tied to this resource
#     sources_for_res = []
#     try:
#         paginator = logs_client.get_paginator("describe_delivery_sources")
#         for page in paginator.paginate():
#             for s in page.get("deliverySources", []):
#                 if resource_arn in s.get("resourceArns", []):
#                     sources_for_res.append(s)
#     except Exception as e:
#         print(f"List delivery sources for {resource_arn}: {e}")
#         return
#     source_names = {s["name"] for s in sources_for_res}
#     # 2. Delete deliveries originating from those sources, collect dest ARNs
#     dest_arns_seen = set()
#     if source_names:
#         try:
#             paginator = logs_client.get_paginator("describe_deliveries")
#             for page in paginator.paginate():
#                 for d in page.get("deliveries", []):
#                     if d.get("deliverySourceName") in source_names:
#                         if d.get("deliveryDestinationArn"):
#                             dest_arns_seen.add(d["deliveryDestinationArn"])
#                         try:
#                             logs_client.delete_delivery(id=d["id"])
#                             print(f"Deleted delivery: {d['id']}")
#                         except Exception as e:
#                             print(f"Delete delivery ({d['id']}): {e}")
#         except Exception as e:
#             print(f"List deliveries: {e}")
#     # 3. Delete destinations — but ONLY ones we own (matched by name prefix
#     #    and not XRAY type) so we never touch the shared XRAY destination.
#     try:
#         all_dests = logs_client.describe_delivery_destinations().get("deliveryDestinations", [])
#         dests_by_arn = {d["arn"]: d for d in all_dests}
#     except Exception as e:
#         print(f"List delivery destinations: {e}")
#         dests_by_arn = {}
#     for dest_arn in dest_arns_seen:
#         d = dests_by_arn.get(dest_arn)
#         if not d:
#             continue
#         if d.get("deliveryDestinationType") == "XRAY":
#             continue  # shared, never delete
#         dname = d.get("name", "")
#         if not dname.startswith(dest_name_prefix):
#             continue  # not ours
#         try:
#             logs_client.delete_delivery_destination(name=dname)
#             print(f"Deleted delivery destination: {dname}")
#         except Exception as e:
#             print(f"Delete delivery destination ({dname}): {e}")
#     # 4. Delete sources
#     for s in sources_for_res:
#         try:
#             logs_client.delete_delivery_source(name=s["name"])
#             print(f"Deleted delivery source: {s['name']}")
#         except Exception as e:
#             print(f"Delete delivery source ({s['name']}): {e}")
#
# # ── Phase 2: Drain Memory CloudWatch log delivery ───────────────────────────
# if MEMORY_ARN:
#     _drain_log_delivery(MEMORY_ARN, dest_name_prefix="mem-")
# else:
#     print("Memory ARN not resolved; skipping memory log delivery cleanup.")
#
# # ── Phase 3: Drain Gateway CloudWatch log delivery (wiring only) ────────────
# if GATEWAY_ARN:
#     _drain_log_delivery(GATEWAY_ARN, dest_name_prefix="gw-")
# else:
#     print("Gateway ARN not resolved; skipping gateway log delivery cleanup.")
#
# # ── Phase 4: Delete Memory and poll until gone ─────────────────────────────
# if MEMORY_ID:
#     try:
#         agentcore_control.delete_memory(memoryId=MEMORY_ID)
#         print(f"Initiated delete of Memory: {MEMORY_ID}")
#         deadline = time.time() + 300
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_memory(memoryId=MEMORY_ID)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Memory fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for Memory to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print(f"Memory {MEMORY_ID} already deleted.")
#     except Exception as e:
#         print(f"Memory delete: {e}")
# else:
#     print(f"Memory {MEMORY_NAME} not found (already deleted).")
#
# # ── Phase 5: Delete CloudWatch log groups ──────────────────────────────────
# log_groups = []
# if MEMORY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/{MEMORY_ID}")
# if GATEWAY_ID:
#     log_groups.append(f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{GATEWAY_ID}")
# for lg in log_groups:
#     try:
#         logs_client.delete_log_group(logGroupName=lg)
#         print(f"Deleted log group: {lg}")
#     except logs_client.exceptions.ResourceNotFoundException:
#         pass
#     except Exception as e:
#         print(f"Log group delete ({lg}): {e}")
#
# # ── Phase 6: Delete IAM Role (used by both harness and memory) ─────────────
# try:
#     for ap in iam.list_attached_role_policies(RoleName=ROLE_NAME).get("AttachedPolicies", []):
#         iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=ap["PolicyArn"])
#     for p in iam.list_role_policies(RoleName=ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=p)
#     iam.delete_role(RoleName=ROLE_NAME)
#     print(f"Deleted IAM role: {ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"IAM role cleanup: {e}")
#
# # ── Phase 7: Delete SSM parameters ─────────────────────────────────────────
# for param in [
#     "harness_id", "harness_arn", "harness_name", "harness_role_arn",
#     "memory_id", "memory_arn", "memory_name",
# ]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#     except Exception as e:
#         print(f"SSM delete ({param}): {e}")
#
# print("\n✅ All Orchestrator resources cleaned up.")